In [ ]:

%pip install \
    transformers==4.27.2 \
    datasets==2.11.0 \
    evaluate==0.4.0 \
    rouge_score==0.1.2 \
    loralib==0.1.1 \
    peft==0.3.0

  Using cached transformers-4.27.2-py3-none-any.whl (6.8 MB)
  Using cached datasets-2.11.0-py3-none-any.whl (468 kB)
  Using cached evaluate-0.4.0-py3-none-any.whl (81 kB)
  Using cached rouge_score-0.1.2-py3-none-any.whl
  Using cached loralib-0.1.1-py3-none-any.whl (8.8 kB)
  Using cached peft-0.3.0-py3-none-any.whl (56 kB)
  Using cached regex-2023.10.3-cp39-cp39-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.13.3-cp39-cp39-win_amd64.whl (3.5 MB)
  Using cached dill-0.3.6-py3-none-any.whl (110 kB)
  Using cached responses-0.18.0-py3-none-any.whl (38 kB)
  Using cached absl_py-2.0.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached nltk-3.8.1-py3-none-any.whl (1.5 MB)
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
  Using cached multiprocess-0.70.14-py39-none-any.whl (132 kB)
Using cached regex-2023.10.3-cp39-cp39-win_amd64.whl (269 kB)
Using cached absl_py-2.0.0-py3-non

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
import torch
import time
import evaluate
import pandas as pd
import numpy as np
from datasets import Dataset

In [ ]:
huggingface_dataset_name = "DATA/paleo_filtered3_s.csv"

df = pd.read_csv(huggingface_dataset_name)
huggingface_dataset = Dataset.from_pandas(df)

huggingface_dataset

Dataset({
    features: ['author', 'published_at', 'updated_at', 'like_count', 'text', 'video', 'summary'],
    num_rows: 9
})

In [ ]:
import torch
torch.cuda.is_available()


True

In [ ]:
torch.cuda.set_device(0)

In [ ]:
#model_name='google/flan-t5-base'
model_name='facebook/bart-large-cnn'
original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Some weights of BartForConditionalGeneration were not initialized from the model checkpoint at facebook/bart-large-cnn and are newly initialized: ['model.shared.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params}\nall model parameters: {all_model_params}\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

print(print_number_of_trainable_model_parameters(original_model))

trainable model parameters: 406290432
all model parameters: 406290432
percentage of trainable model parameters: 100.00%


In [ ]:
def tokenize_function(example):
    start_prompt = 'Summarize the following conversation.\n\n'
    end_prompt = '\n\nSummary: '
    prompt = [start_prompt + dialogue + end_prompt for dialogue in example["text"]]
    example['input_ids'] = tokenizer(prompt, padding="max_length", truncation=True, return_tensors="pt").input_ids
    example['labels'] = tokenizer(example["summary"], padding="max_length", truncation=True, return_tensors="pt").input_ids

    return example

# The dataset actually contains 3 diff splits: train, validation, test.
# The tokenize_function code is handling all data across all splits in batches.
tokenized_datasets = huggingface_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/9 [00:00<?, ? examples/s]

In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(['author', 'published_at', 'updated_at', 'like_count', 'text', 'video', 'summary',])

In [ ]:
print(tokenized_datasets)

Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 9
})


In [ ]:
output_dir = f'./results3'

In [ ]:
L_RATE = 1e-5
BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH = 2
WEIGHT_DECAY = 0.01
SAVE_TOTAL_LIM = 3
NUM_EPOCHS = 3

# Set up training arguments
training_args = TrainingArguments(
   output_dir="./results3",
   learning_rate=L_RATE,
   per_device_train_batch_size=BATCH_SIZE,
   weight_decay=WEIGHT_DECAY,
   save_total_limit=SAVE_TOTAL_LIM,
   num_train_epochs=NUM_EPOCHS,
   gradient_accumulation_steps=4,
   fp16=True,
   push_to_hub=False
)



In [ ]:
trainer = Trainer(
   model=original_model,
   args=training_args,
   train_dataset=tokenized_datasets,
   tokenizer=tokenizer,

)

In [ ]:
trainer.train()

C:\Users\shahe\Anaconda3\envs\torch\lib\site-packages\transformers\optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
You're using a BartTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


OutOfMemoryError: CUDA out of memory. Tried to allocate 64.00 MiB. GPU 0 has a total capacty of 4.00 GiB of which 0 bytes is free. Of the allocated memory 6.99 GiB is allocated by PyTorch, and 17.79 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
trainer.save_model(output_dir)

In [ ]:
last_checkpoint = "./results3"

In [ ]:
original_model = AutoModelForSeq2SeqLM.from_pretrained(last_checkpoint, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(last_checkpoint)